## 위치·연계

- **이 노트북:** `ai/radar-cruw/0_cruw_radar_training.ipynb`
- **데이터 루트(기본):** `ai/radar-cruw/data/` — CRUW 받은 zip·폴더를 여기 두면 노트북이 자동 탐색합니다 (`CRUW_DATA_DIR`로 덮어쓰기 가능).
- **VoD:** 수신 후 **`vod-devkit/`** (`1_frame_information.ipynb`, `PP-Radar.md`)와 파이프라인·좌표를 맞추면 됩니다.
- **추론 서버:** **`ai-inference/`** 는 별도(FastAPI 등). 이 노트북은 **학습/데이터 프로토타입** 전용입니다.

### 이 레포에서의 `ai/` 구조 (요약)

```
hanhwa_final/
├── ai/
│   ├── README.md
│   └── radar-cruw/
│       ├── 0_cruw_radar_training.ipynb   ← 이 파일
│       ├── requirements-cruw.txt
│       └── data/                         ← CRUW 파일 배치 (git에 없을 수 있음)
│           ├── TRAIN_RAD_H-001.zip       ← 학습 RAMap (압축 그대로 가능)
│           ├── …/TRAIN_RAD_H_ANNO/**/*.txt
│           └── TEST_RAD_H-003/.../RADAR_RA_H/*.npy
├── vod-devkit/
└── ai-inference/
```

**실행 순서:** 아래 **경로 셀(셀 2)** → **파싱·로드 셀** → (선택) JSON → **PyTorch 셀** → **Dataset 셀**.

---

# CRUW 레이더 학습 준비 (VoD 지연 시 대안)

이 노트북은 **CRUW (Camera Radar With You)** 공개 데이터셋으로 레이더 기반 인지 모델을 실험하기 위한 **참고 문서 목록**, **데이터 구조 이해**, **경량 PyTorch 학습 루프 예시**를 담습니다. 본격적인 SOTA 재현은 공식 **RODNet** 저장소의 학습 스크립트를 사용하는 것이 좋습니다.

---

## CRUW 관련 공식·논문·코드 (서류/리소스)

| 구분 | 링크 | 비고 |
|------|------|------|
| 데이터셋 홈 | [https://www.cruwdataset.org/](https://www.cruwdataset.org/) | 소개, 다운로드 안내 |
| Introduction | [https://www.cruwdataset.org/introduction](https://www.cruwdataset.org/introduction) | 센서·시나리오·어노테이션 개요 |
| Resources (Devkit, RODNet) | [https://www.cruwdataset.org/resources](https://www.cruwdataset.org/resources) | **cruw-devkit**, **RODNet** GitHub |
| **cruw-devkit** | [https://github.com/yizhou-wang/cruw-devkit](https://github.com/yizhou-wang/cruw-devkit) | 캘리브레이션, RAMap↔좌표 변환, 시각화, ROD2021 튜토리얼 노트북 |
| **RODNet** (학습/추론 코드) | [https://github.com/yizhou-wang/RODNet](https://github.com/yizhou-wang/RODNet) | Range–Azimuth 히트맵 입력, 교차 모달 학습 |
| RODNet WACV 2021 | [https://openaccess.thecvf.com/content/WACV2021/html/Wang_RODNet_Radar_Object_Detection_Using_Cross-Modal_Supervision_WACV_2021_paper.html](https://openaccess.thecvf.com/content/WACV2021/html/Wang_RODNet_Radar_Object_Detection_Using_Cross-Modal_Supervision_WACV_2021_paper.html) | 회의 논문 |
| RODNet IEEE J-STSP (저널) | [https://ieeexplore.ieee.org/document/9353210](https://ieeexplore.ieee.org/document/9353210) / [arXiv:2102.05150](https://arxiv.org/abs/2102.05150) | 확장판 |
| CRUW·데이터셋 논문 (CVPRW 2021) | [WAD paper (OpenAccess)](https://openaccess.thecvf.com/content/CVPR2021W/WAD/html/Wang_Rethinking_of_Radars_Role_A_Camera-Radar_Dataset_and_Systematic_Annotator_CVPRW_2021_paper.html) | 데이터셋·어노테이터 체계 |
| ROD2021 Challenge | [CodaLab evaluation](https://codalab.lisn.upsaclay.fr/competitions/1063) | 벤치마크 제출(2022 재개 공지 있음) |
| Devkit Colab 튜토리얼 | [cruw_devkit_tutorial_rod2021.ipynb](https://colab.research.google.com/github/yizhou-wang/cruw-devkit/blob/master/tutorials/cruw_devkit_tutorial_rod2021.ipynb) | ROD2021 형식 읽기·평가 |

**라이선스:** CRUW는 [cruw-devkit README](https://github.com/yizhou-wang/cruw-devkit)에 명시된 대로 학술·비영리 연구용 제한이 있습니다. 사용 전 사이트 약관을 확인하세요.

---

## 원하시는 출력과 CRUW/레이더에서의 대응

| 목표 | CRUW에서의 일반적 대응 |
|------|------------------------|
| 표적 감지 | RAMap 상 객체 존재/중심 (RODNet 검출 또는 히트맵 회귀) |
| 표적까지 거리 | **range(m)** 또는 range 축 인덱스 → 거리 그리드와 캘리브레이션으로 환산 |
| 좌측/정면/우측 | **azimuth(rad)** 또는 azimuth 축 인덱스 (차량 전방 기준 각도 구간으로 구분 가능) |
| 접근 중 / 이탈 중 | 원시 chirp 스택에서 도플러(속도) 정보 추출·학습 — RODNet의 multi-chirp 경로 또는 **range–doppler** 표현 추가 모델링이 필요할 수 있음 |
| 이동 경로 | 시계열 RAMap 또는 검출 (range, azimuth) 시퀀스로 **궤적** 근사 (별도 트래킹/연속 프레임 손실) |

**데이터:** 기본 경로는 **`ai/radar-cruw/data/`** 입니다. ROD2021 라벨은 `**/TRAIN_RAD_H_ANNO/**/*.txt`, RAMap은 **`TRAIN_RAD_H*.zip`** 안 또는 `**/RADAR_RA_H/*.npy`. General CRUW **JSON** 메타는 별도 배포 시에만 해당합니다 ([devkit Annotation Format](https://github.com/yizhou-wang/cruw-devkit)).


## 환경 설정 (선택)

```bash
# 예: 별도 conda 환경
conda create -n cruw python=3.10 -y
conda activate cruw
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124  # GPU에 맞게 조정
pip install numpy matplotlib tqdm

# 데이터 도구 (로컬 클론 후)
# git clone https://github.com/yizhou-wang/cruw-devkit.git
# cd cruw-devkit && pip install -e .
```

기본값은 **`ai/radar-cruw/data`** 입니다. 다른 위치면 환경 변수 `CRUW_DATA_DIR` 또는 아래 셀에서 개별 경로를 지정하세요.

In [31]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Optional

import numpy as np


import io
import zipfile


def find_repo_root() -> Path:
    """노트북 cwd 기준 hanhwa_final 루트. vod-devkit / ai/radar-cruw 마커로 탐지."""
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "vod-devkit").is_dir():
            return cand
        if (cand / "ai" / "radar-cruw" / "requirements-cruw.txt").is_file():
            return cand
        if (cand / "ai" / "radar-cruw" / "data").is_dir():
            return cand
    return p


REPO_ROOT = find_repo_root()

# ========= 프로젝트 기본 경로: ai/radar-cruw/data =========
DATA_DIR = (
    Path(os.environ["CRUW_DATA_DIR"]).resolve()
    if os.environ.get("CRUW_DATA_DIR")
    else REPO_ROOT / "ai" / "radar-cruw" / "data"
)


def collect_anno_by_stem(data_dir: Path) -> dict[str, Path]:
    """TRAIN_RAD_H_ANNO 내 .txt 를 시퀀스 stem -> 경로 로 정리 (중복 stem 은 하나만)."""
    out: dict[str, Path] = {}
    if not data_dir.is_dir():
        return out
    for p in sorted(data_dir.glob("**/TRAIN_RAD_H_ANNO/**/*.txt")):
        out[p.stem] = p
    return out


_TRAIN_ZIP_INDEX: Optional[list[tuple[Path, set[str]]]] = None


def train_zip_stem_index(data_dir: Path) -> list[tuple[Path, set[str]]]:
    """TRAIN_RAD_H*.zip 마다 RAMap이 있는 시퀀스 stem 집합 (한 번만 훑음)."""
    global _TRAIN_ZIP_INDEX
    if _TRAIN_ZIP_INDEX is not None:
        return _TRAIN_ZIP_INDEX
    out: list[tuple[Path, set[str]]] = []
    for zp in sorted(data_dir.glob("**/TRAIN_RAD_H*.zip")):
        with zipfile.ZipFile(zp, "r") as z:
            stems: set[str] = set()
            for n in z.namelist():
                n2 = n.replace("\\", "/")
                if "/RADAR_RA_H/" in n2 and n.endswith(".npy"):
                    parts = n2.split("/")
                    if len(parts) >= 3 and parts[0] == "TRAIN_RAD_H":
                        stems.add(parts[1])
        if stems:
            out.append((zp, stems))
    _TRAIN_ZIP_INDEX = out
    return out


def find_radar_source_for_sequence(data_dir: Path, stem: str) -> tuple:
    """RAMap 소스: ('dir', RADAR_RA_H 경로) | ('zip', zip 경로, TRAIN_RAD_H/<stem>/RADAR_RA_H) | ('none',)."""
    for d in data_dir.glob(f"**/{stem}/RADAR_RA_H"):
        if d.is_dir() and d.name == "RADAR_RA_H" and any(d.glob("*.npy")):
            return ("dir", d)
    prefix = f"TRAIN_RAD_H/{stem}/RADAR_RA_H"
    for zp, stems in train_zip_stem_index(data_dir):
        if stem not in stems:
            continue
        return ("zip", zp, prefix)
    return ("none",)


def stems_with_radar_available(data_dir: Path) -> set[str]:
    """zip 인덱스 + 풀린 RADAR_RA_H 폴더에서 stem 한 번에 수집."""
    s: set[str] = set()
    for _, stems in train_zip_stem_index(data_dir):
        s |= stems
    for d in data_dir.glob("**/RADAR_RA_H"):
        if d.is_dir() and d.name == "RADAR_RA_H" and any(d.glob("*.npy")):
            s.add(d.parent.name)
    return s


def list_sequences_with_radar(data_dir: Path, annos: dict[str, Path]) -> list[str]:
    """라벨 stem 중 RAMap(폴더 또는 TRAIN_RAD zip)이 있는 것만."""
    avail = stems_with_radar_available(data_dir)
    return sorted(stem for stem in annos if stem in avail)


anno_by_stem = collect_anno_by_stem(DATA_DIR)
CRUW_SEQUENCE_STEM = os.environ.get("CRUW_SEQUENCE", "").strip()
CRUW_ANNO_TXT: Optional[Path] = Path(os.environ["CRUW_ANNO_TXT"]).resolve() if os.environ.get("CRUW_ANNO_TXT") else None
RADAR_SOURCE: tuple = ("none",)

if os.environ.get("CRUW_RADAR_DIR"):
    _rd = Path(os.environ["CRUW_RADAR_DIR"]).resolve()
    if _rd.is_dir():
        RADAR_SOURCE = ("dir", _rd)

if CRUW_ANNO_TXT is not None:
    CRUW_SEQUENCE_STEM = CRUW_SEQUENCE_STEM or CRUW_ANNO_TXT.stem
elif CRUW_SEQUENCE_STEM and CRUW_SEQUENCE_STEM in anno_by_stem:
    CRUW_ANNO_TXT = anno_by_stem[CRUW_SEQUENCE_STEM]

if RADAR_SOURCE[0] == "none" and CRUW_SEQUENCE_STEM:
    RADAR_SOURCE = find_radar_source_for_sequence(DATA_DIR, CRUW_SEQUENCE_STEM)

if CRUW_ANNO_TXT is None or RADAR_SOURCE[0] == "none":
    matched = list_sequences_with_radar(DATA_DIR, anno_by_stem)
    pick = ""
    if CRUW_SEQUENCE_STEM and CRUW_SEQUENCE_STEM in matched:
        pick = CRUW_SEQUENCE_STEM
    elif matched:
        pick = matched[0]
    if pick:
        CRUW_SEQUENCE_STEM = pick
        CRUW_ANNO_TXT = anno_by_stem.get(pick)
        RADAR_SOURCE = find_radar_source_for_sequence(DATA_DIR, pick)

PAIR_OK = bool(
    CRUW_ANNO_TXT and CRUW_ANNO_TXT.is_file() and RADAR_SOURCE and RADAR_SOURCE[0] != "none"
)

# 레거시 변수: 폴더 기반 로더만 쓸 때
CRUW_RADAR_DIR: Optional[Path] = RADAR_SOURCE[1] if RADAR_SOURCE[0] == "dir" else None

# (선택) General CRUW JSON
SEQUENCE_JSON: Optional[Path] = (
    Path(os.environ["CRUW_SEQUENCE_JSON"]).resolve() if os.environ.get("CRUW_SEQUENCE_JSON") else None
)

MATCHED_STEMS = list_sequences_with_radar(DATA_DIR, anno_by_stem) if anno_by_stem else []

print("REPO_ROOT          :", REPO_ROOT)
print("DATA_DIR           :", DATA_DIR, "| exists:", DATA_DIR.is_dir())
print("라벨+RAMap 짝 가능 시퀀스 수:", len(MATCHED_STEMS), "| 예시:", MATCHED_STEMS[:5])
print("CRUW_SEQUENCE_STEM :", CRUW_SEQUENCE_STEM or "(없음)")
print("CRUW_ANNO_TXT      :", CRUW_ANNO_TXT)
print("RADAR_SOURCE       :", RADAR_SOURCE[0], RADAR_SOURCE[1:] if len(RADAR_SOURCE) > 1 else "")
print("PAIR_OK (라벨↔RAMap):", PAIR_OK)
print("SEQUENCE_JSON      :", SEQUENCE_JSON or "(미사용)")

REPO_ROOT          : C:\Users\taehu\Desktop\projects\hanhwa_final
DATA_DIR           : C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\data | exists: True
라벨+RAMap 짝 가능 시퀀스 수: 40 | 예시: ['2019_04_09_BMS1000', '2019_04_09_BMS1001', '2019_04_09_BMS1002', '2019_04_09_CMS1002', '2019_04_09_PMS1000']
CRUW_SEQUENCE_STEM : 2019_04_09_BMS1000
CRUW_ANNO_TXT      : C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\data\drive-download-20260325T021904Z-1-004\TRAIN_RAD_H_ANNO\TRAIN_RAD_H_ANNO\2019_04_09_BMS1000.txt
RADAR_SOURCE       : dir (WindowsPath('C:/Users/taehu/Desktop/projects/hanhwa_final/ai/radar-cruw/data/TRAIN_RAD_H-001/TRAIN_RAD_H/2019_04_09_BMS1000/RADAR_RA_H'),)
PAIR_OK (라벨↔RAMap): True
SEQUENCE_JSON      : (미사용)


### ROD2021 텍스트 라벨 + `ai/radar-cruw/data/` 배치

**위 경로 셀( `DATA_DIR` / `PAIR_OK` )을 먼저 실행한 뒤** 이어서 실행하세요.

- **라벨:** 한 줄에 `frame_id  range(m)  azimuth(rad)  class_name` ([cruw-devkit ROD2021](https://github.com/yizhou-wang/cruw-devkit)).
- **RAMap:** `RADAR_RA_H/000000_0000.npy` 형식 — 로컬에서 확인한 바 **`(128, 128, 2)`** → 모델용 **`[2, 128, 128]`** 로 `permute` 합니다.

**폴더 예시**

```
ai/radar-cruw/data/
  drive-download-.../TRAIN_RAD_H_ANNO/TRAIN_RAD_H_ANNO/*.txt
  TEST_RAD_H-003/TEST_RAD_H/<시퀀스명>/RADAR_RA_H/*.npy
```

**짝 맞추기:** 학습 RAMap은 보통 **`TRAIN_RAD_H-001.zip`** 안에 있습니다. 압축을 풀지 않아도 노트북이 zip에서 읽습니다. 라벨 `*.txt` 파일 stem(예: `2019_04_09_BMS1000`)과 동일한 폴더/`TRAIN_RAD_H/<stem>/RADAR_RA_H`가 있어야 합니다. 환경 변수: `CRUW_SEQUENCE`(stem), `CRUW_ANNO_TXT`, `CRUW_RADAR_DIR`, `CRUW_DATA_DIR`.

In [32]:
import io
import zipfile

# 셀 2에서 RADAR_SOURCE, PAIR_OK, CRUW_* 가 정의되어 있어야 합니다.
def parse_rod2021_txt(txt_path: Path) -> dict[int, list[tuple[float, float, str]]]:
    """ROD2021: frame_id -> [(range_m, azimuth_rad, class_name), ...]"""
    by_frame: dict[int, list[tuple[float, float, str]]] = {}
    if not txt_path.is_file():
        return by_frame
    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 4:
                continue
            fid = int(parts[0])
            r_m, az_rad = float(parts[1]), float(parts[2])
            cls = parts[3]
            by_frame.setdefault(fid, []).append((r_m, az_rad, cls))
    return by_frame


def radar_npy_path(radar_dir: Path, frame_id: int, chunk_index: int = 0) -> Path:
    return radar_dir / f"{frame_id:06d}_{chunk_index:04d}.npy"


_ZIP_HANDLES: dict[str, zipfile.ZipFile] = {}


def _zip_get(zp: Path) -> zipfile.ZipFile:
    key = str(zp.resolve())
    if key not in _ZIP_HANDLES:
        _ZIP_HANDLES[key] = zipfile.ZipFile(zp, "r")
    return _ZIP_HANDLES[key]


def load_ramap_rood2021(radar_source: tuple, frame_id: int, chunk_index: int = 0) -> Optional[np.ndarray]:
    """폴더 RADAR_RA_H 또는 TRAIN_RAD_H*.zip 안의 RAMap. 기본 청크 `_0000`."""
    if not radar_source or radar_source[0] == "none":
        return None
    fn = f"{frame_id:06d}_{chunk_index:04d}.npy"
    kind = radar_source[0]
    if kind == "dir":
        d = radar_source[1]
        if not d.is_dir():
            return None
        p = d / fn
        if p.is_file():
            return np.load(p)
        matches = sorted(d.glob(f"{frame_id:06d}_*.npy"))
        return np.load(matches[0]) if matches else None
    if kind == "zip":
        _, zp, prefix = radar_source
        prefix = prefix.replace("\\", "/")
        inner = f"{prefix}/{fn}"
        zf = _zip_get(zp)
        names = zf.namelist()
        if inner not in names:
            inner_alt = inner.replace("/", "\\")
            if inner_alt in names:
                inner = inner_alt
            else:
                hits = sorted(
                    n
                    for n in names
                    if n.replace("\\", "/").startswith(prefix + "/")
                    and f"{frame_id:06d}_" in n
                    and n.endswith(".npy")
                )
                if not hits:
                    return None
                inner = hits[0]
        return np.load(io.BytesIO(zf.read(inner)))
    return None


def ramap_to_chw(arr: np.ndarray) -> np.ndarray:
    """(H,W,C) RAMap -> [C,H,W]. (C,H,W) 는 그대로."""
    if arr.ndim == 2:
        return arr[np.newaxis, ...]
    if arr.ndim == 3:
        if arr.shape[0] <= 32 and arr.shape[-1] > 32:
            return arr
        return np.transpose(arr, (2, 0, 1))
    return arr


def range_az_to_pixel(
    range_m: float,
    az_rad: float,
    h: int,
    w: int,
    range_max_m: float = 50.0,
    az_fov_rad: float = 1.4,
) -> tuple[float, float]:
    """거리·방위 → 히트맵 row/col. FOV는 [-az_fov/2,+az_fov/2] 근사(필요 시 상수 조정)."""
    half = az_fov_rad / 2.0
    cy = float(np.clip((range_m / range_max_m) * (h - 1), 0, h - 1))
    ax = float(np.clip((az_rad + half) / az_fov_rad, 0.0, 1.0))
    cx = ax * (w - 1)
    return cy, cx


by_frame = parse_rod2021_txt(CRUW_ANNO_TXT) if CRUW_ANNO_TXT else {}
print("ROD2021 라벨 프레임 수:", len(by_frame), "| 예시 frame_id:", sorted(by_frame.keys())[:8])

if by_frame and PAIR_OK:
    _fid = min(by_frame.keys())
    _arr = load_ramap_rood2021(RADAR_SOURCE, _fid)
    if _arr is not None:
        print("시퀀스:", CRUW_SEQUENCE_STEM, "| frame", _fid, "RAMap", _arr.shape, "-> CHW:", ramap_to_chw(_arr).shape)
    else:
        print("경고: frame", _fid, "에 해당 .npy 없음")
elif not PAIR_OK:
    print("PAIR_OK=False — 라벨과 RAMap 짝을 확인하세요 (TRAIN_RAD_H*.zip 내 시퀀스와 TRAIN_RAD_H_ANNO stem 이 일치해야 함).")
else:
    print("라벨 없음")

ROD2021 라벨 프레임 수: 897 | 예시 frame_id: [0, 1, 2, 3, 4, 5, 6, 7]
시퀀스: 2019_04_09_BMS1000 | frame 0 RAMap (128, 128, 2) -> CHW: (2, 128, 128)


### (선택) General CRUW `metadata.json`

별도 JSON 메타가 있으면 `CRUW_SEQUENCE_JSON`으로 경로를 주고 아래 셀을 실행합니다. **이 프로젝트 기본 흐름은 위 ROD2021 `.txt` + `RADAR_RA_H` 입니다.**

In [33]:
import json


def load_sequence_metadata(json_path: Optional[Path]) -> Optional[dict]:
    if not json_path or not json_path.is_file():
        return None
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)


meta_json = load_sequence_metadata(SEQUENCE_JSON)
if meta_json:
    print("JSON dataset:", meta_json.get("dataset"), meta_json.get("seq_name"), "n_frames:", meta_json.get("n_frames"))
else:
    print("SEQUENCE_JSON 미설정 또는 파일 없음 (ROD2021만 쓰면 생략 가능)")

SEQUENCE_JSON 미설정 또는 파일 없음 (ROD2021만 쓰면 생략 가능)


---

## 경량 학습 예시: RAMap → 객체 히트맵 (PyTorch)

실제 RODNet은 여러 chirp·시퀀스와 복잡한 디코더를 사용합니다. 여기서는 **동일한 아이디어**(공간 히트맵 학습)를 최소 구현으로 보여, 데이터 연결 후 손쉽게 확장할 수 있게 합니다.

- 입력: `[B, C, H_range, W_azimuth]` — 로컬 RAMap은 **`(128,128,2)` → `C=2`**
- 출력: 각 객체 클래스별 2D confidence 맵 (여기서는 단일 채널 가우시안 타깃)

**본격 학습:** [RODNet 저장소](https://github.com/yizhou-wang/RODNet)의 `train.py` 및 설정 파일을 사용하세요.

In [34]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


def gaussian_heatmap(h: int, w: int, cy: float, cx: float, sigma: float = 2.0) -> np.ndarray:
    yy, xx = np.ogrid[:h, :w]
    g = np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * sigma**2))
    return g.astype(np.float32)


class DummyRAMapDataset(Dataset):
    """CRUW 미다운로드 시: 가짜 RAMap + 가짜 중심으로 학습 루프 검증 (채널=2는 data RAMap과 동일)."""

    def __init__(self, n: int = 256, h: int = 128, w: int = 128, n_chirp: int = 2):
        self.n = n
        self.h, self.w = h, w
        self.n_chirp = n_chirp

    def __len__(self) -> int:
        return self.n

    def __getitem__(self, idx: int):
        rng = np.random.default_rng(idx)
        x = rng.normal(size=(self.n_chirp, self.h, self.w)).astype(np.float32)
        cy = float(rng.uniform(8, self.h - 8))
        cx = float(rng.uniform(8, self.w - 8))
        target = gaussian_heatmap(self.h, self.w, cy, cx, sigma=3.0)
        return torch.from_numpy(x), torch.from_numpy(target[None, ...])  # [C,H,W], [1,H,W]


class TinyRadarHeatmapNet(nn.Module):
    def __init__(self, in_ch: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def train_one_epoch(model, loader, opt, criterion):
    model.train()
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        opt.step()
        total += loss.item() * xb.size(0)
    return total / len(loader.dataset)


BATCH = 8
EPOCHS = 3
ds = DummyRAMapDataset(n=128, h=128, w=128, n_chirp=2)
loader = DataLoader(ds, batch_size=BATCH, shuffle=True, num_workers=0)
model = TinyRadarHeatmapNet(in_ch=2).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

for ep in range(EPOCHS):
    loss = train_one_epoch(model, loader, opt, criterion)
    print(f"epoch {ep+1}/{EPOCHS}  mse_loss={loss:.6f}")

print("더미 학습 완료. CRUW RAMap을 `Dataset`에 연결하면 동일 루프로 치환 가능합니다.")

device: cpu
epoch 1/3  mse_loss=0.087426
epoch 2/3  mse_loss=0.004238
epoch 3/3  mse_loss=0.002083
더미 학습 완료. CRUW RAMap을 `Dataset`에 연결하면 동일 루프로 치환 가능합니다.


### 실데이터 연결: `CruwRod2021HeatmapDataset` (선택)

위 **`gaussian_heatmap`** 이 있는 셀을 실행한 뒤 이 셀을 실행하세요. **`RADAR_SOURCE`**(폴더 또는 zip)와 **`by_frame`**으로 RAMap을 읽습니다. `PAIR_OK`가 True일 때만 라벨과 RAMap이 같은 시퀀스입니다.

In [35]:
import torch
from torch.utils.data import Dataset

# `gaussian_heatmap`, `load_ramap_rood2021`, `ramap_to_chw`, `range_az_to_pixel` 는 상위 셀 정의 필요


class CruwRod2021HeatmapDataset(Dataset):
    """ROD2021 txt(by_frame) + RADAR_SOURCE(폴더 또는 TRAIN zip)."""

    def __init__(
        self,
        by_frame: dict[int, list[tuple[float, float, str]]],
        radar_source: tuple,
        range_max_m: float = 50.0,
        az_fov_rad: float = 1.4,
        sigma_px: float = 2.5,
        h: int = 128,
        w: int = 128,
    ):
        self.by_frame = by_frame
        self.radar_source = radar_source
        self.range_max_m = range_max_m
        self.az_fov_rad = az_fov_rad
        self.sigma_px = sigma_px
        self.h, self.w = h, w
        self.frame_ids = sorted(by_frame.keys())

    def __len__(self) -> int:
        return len(self.frame_ids)

    def __getitem__(self, idx: int):
        fid = self.frame_ids[idx]
        raw = load_ramap_rood2021(self.radar_source, fid)
        if raw is None:
            raw = np.zeros((self.h, self.w, 2), dtype=np.float32)
        x = torch.from_numpy(ramap_to_chw(raw).astype(np.float32))

        tgt = np.zeros((self.h, self.w), dtype=np.float32)
        for r_m, az_rad, _ in self.by_frame[fid]:
            cy, cx = range_az_to_pixel(
                r_m, az_rad, self.h, self.w, self.range_max_m, self.az_fov_rad
            )
            tgt = np.maximum(tgt, gaussian_heatmap(self.h, self.w, cy, cx, sigma=self.sigma_px))
        y = torch.from_numpy(tgt[None, ...])
        return x, y


if by_frame and PAIR_OK and RADAR_SOURCE[0] != "none":
    try:
        _ds_real = CruwRod2021HeatmapDataset(by_frame, RADAR_SOURCE)
        print("CruwRod2021HeatmapDataset len =", len(_ds_real))
        x0, y0 = _ds_real[0]
        print("샘플 x:", tuple(x0.shape), "y:", tuple(y0.shape))
    except Exception as e:
        print("CruwRod2021HeatmapDataset 스킵:", e)
else:
    print("PAIR_OK=False 이거나 라벨 없음 — 셀 2 경로·시퀀스 확인")

CruwRod2021HeatmapDataset len = 897
샘플 x: (2, 128, 128) y: (1, 128, 128)


### 접근/이탈·경로 표시를 모델에 넣을 때 (설계 메모)

1. **도플러(접근/이탈):** CRUW의 `.npy`가 chirp 차원을 포함하면, 시간축 FFT 또는 모델 내 **clutter map 대비 차분**으로 속도 부호를 학습할 수 있습니다. VoD의 `v_r`와 유사한 역할입니다.
2. **경로:** 연속 프레임에서 검출 (range_idx, az_idx) 또는 (range_m, az_rad)를 추출한 뒤 **Kalman 필터** 또는 **LSTM/Transformer**로 보정된 궤적을 그리면 시각화에 적합합니다.
3. **RODNet 연동:** 학습은 저장소 클론 후 `python`이 아닌 해당 repo의 스크립트가 표준이므로, 이 노트북은 **데이터 탐색 + 프로토타입**용으로 두고, 재현은 RODNet 쪽을 권장합니다.

```bash
# 예시 (RODNet 저장소 구조는 버전마다 다를 수 있음 — README 확인)
# git clone https://github.com/yizhou-wang/RODNet.git && cd RODNet
# pip install -r requirements.txt
# python train.py --config ...
```